# 01 — Setup Tables

Load `./app/data/source/orders.csv` into **four table formats** side-by-side:

| # | Format | Catalog / Location |
|---|--------|--------------------|
| 1 | **Iceberg** | `iceberg_catalog.my_database.orders_iceberg` |
| 2 | **Delta** | `.tmp/local_delta_warehouse/my_database/orders_delta` |
| 3 | **Hudi** | `.tmp/local_hudi_warehouse/my_database/orders_hudi` |
| 4 | **Parquet** | `.tmp/local_parquet_warehouse/my_database/orders_parquet` |

> **Pre-requisite:** `spark-defaults.conf` must include Iceberg, Delta, and Hudi packages + extensions.

In [1]:
from app.utils.spark_session import spark, display_df

spark

---
## Step 1 — Read CSV with explicit schema

In [7]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("order_id",      IntegerType(), nullable=False),
    StructField("customer_name", StringType(),  nullable=False),
    StructField("product",       StringType(),  nullable=False),
    StructField("quantity",      IntegerType(), nullable=False),
    StructField("unit_price",    DoubleType(),  nullable=False),
    StructField("order_date",    StringType(),  nullable=False),
    StructField("status",        StringType(),  nullable=False),
])

orders_df = spark.read.csv("./app/data/source/orders.csv", header=True, schema=schema)
orders_df.printSchema()
orders_df.show(truncate=False)

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pendin

In [9]:
display(orders_df)

order_id,customer_name,product,quantity,unit_price,order_date,status
1,Alice Johnson,Laptop,1,1299.99,2024-01-15,completed
2,Bob Smith,Mouse,2,29.99,2024-01-16,completed
3,Carol White,Keyboard,1,79.99,2024-01-17,pending
4,David Lee,Monitor,1,399.99,2024-01-18,completed
5,Eve Martinez,Headphones,3,149.99,2024-01-19,shipped
6,Frank Brown,Webcam,2,89.99,2024-01-20,pending
7,Grace Kim,USB Hub,1,49.99,2024-01-21,completed
8,Henry Wilson,Laptop Stand,2,59.99,2024-01-22,shipped
9,Iris Chen,SSD Drive,1,119.99,2024-01-23,completed
10,James Davis,Microphone,1,199.99,2024-01-24,pending


---
## 1 — Iceberg

- Open table format with **ACID transactions** and **time travel**
- Catalog: `iceberg_catalog` (Hadoop catalog, configured in `spark-defaults.conf`)
- Unique feature: **snapshot-based versioning**

In [3]:
orders_df.writeTo("iceberg_catalog.my_database.orders_iceberg") \
    .tableProperty("format-version", "2") \
    .createOrReplace()

print("Written to Iceberg: iceberg_catalog.my_database.orders_iceberg")

Written to Iceberg: iceberg_catalog.my_database.orders_iceberg


In [4]:
iceberg_df = spark.table("iceberg_catalog.my_database.orders_iceberg")
iceberg_df.show(truncate=False)
print(f"Rows: {iceberg_df.count()}")

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10      |James Davis  |Microphone  |1       |199.99

In [5]:
# Snapshot history (time travel feature)
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM iceberg_catalog.my_database.orders_iceberg.snapshots
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|1547481985620889591|2026-04-28 19:35:55.239|overwrite|
|3789211374125154207|2026-04-28 19:36:40.871|overwrite|
|1849591807437829809|2026-04-28 19:40:16.681|overwrite|
|3197429995744435950|2026-04-28 19:41:33.963|overwrite|
|3629707673483375348|2026-04-28 19:41:59.276|overwrite|
|3228785992298635848|2026-04-28 19:43:23.816|overwrite|
|4589610244947835607|2026-04-28 20:13:10.872|overwrite|
+-------------------+-----------------------+---------+



---
## 2 — Delta

- Delta Lake with **ACID transactions** and **schema enforcement**
- Path-based write; reads via `spark.read.format("delta")`
- Unique feature: **transaction log** stored in `_delta_log/`

In [57]:
DELTA_PATH = ".tmp/local_delta_warehouse/my_database/orders_delta"

orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(DELTA_PATH)

print(f"Written to Delta: {DELTA_PATH}")

Written to Delta: .tmp/local_delta_warehouse/my_database/orders_delta


In [58]:
delta_df = spark.read.format("delta").load(DELTA_PATH)
delta_df.show(truncate=False)
print(f"Rows: {delta_df.count()}")

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10      |James Davis  |Microphone  |1       |199.99

In [59]:
# Register Delta path as a cataloged table so DESCRIBE HISTORY works via Spark Connect
# LOCATION needs absolute path (/app is the container CWD where volumes are mounted)
DELTA_ABS_PATH = f"/app/{DELTA_PATH}"

spark.sql("DROP TABLE IF EXISTS spark_catalog.default.orders_delta")
spark.sql(f"""
    CREATE TABLE spark_catalog.default.orders_delta
    USING delta
    LOCATION '{DELTA_ABS_PATH}'
""")

spark.sql("DESCRIBE HISTORY spark_catalog.default.orders_delta") \
    .select("version", "timestamp", "operation") \
    .show(truncate=False)

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|5      |2026-04-28 19:43:25.042|WRITE    |
|4      |2026-04-28 19:43:08.463|WRITE    |
|3      |2026-04-28 19:42:01.352|WRITE    |
|2      |2026-04-28 19:40:22.852|WRITE    |
|1      |2026-04-28 19:36:43.334|WRITE    |
|0      |2026-04-28 19:36:00.397|WRITE    |
+-------+-----------------------+---------+



---
## 3 — Hudi

- Apache Hudi with **upserts** and **incremental processing**
- Path-based table (no catalog required)
- Unique feature: **record-level upserts** via `recordkey.field`

> **Setup:** Add to `spark-defaults.conf`:
> ```
> spark.jars.packages   ...,org.apache.hudi:hudi-spark3.5-bundle_2.13:1.0.2
> spark.sql.extensions  ...,org.apache.spark.sql.hudi.HoodieSparkSessionExtension
> ```
> Check [Hudi releases](https://hudi.apache.org/releases/) for the bundle matching your Spark version.

In [60]:
# HUDI_PATH = ".tmp/local_hudi_warehouse/my_database/orders_hudi"

# hudi_options = {
#     "hoodie.table.name":                       "orders_hudi",
#     "hoodie.datasource.write.recordkey.field":  "order_id",
#     "hoodie.datasource.write.precombine.field": "order_date",
#     "hoodie.datasource.write.operation":        "bulk_insert",
#     "hoodie.datasource.write.table.type":       "COPY_ON_WRITE",
# }

# orders_df.write \
#     .format("hudi") \
#     .options(**hudi_options) \
#     .mode("overwrite") \
#     .save(HUDI_PATH)

# print(f"Written to Hudi: {HUDI_PATH}")

In [61]:
# # Hudi adds internal metadata columns — select only the original ones
# original_cols = ["order_id", "customer_name", "product", "quantity", "unit_price", "order_date", "status"]

# hudi_df = spark.read.format("hudi").load(HUDI_PATH)
# hudi_df.select(original_cols).show(truncate=False)
# print(f"Rows: {hudi_df.count()}")

---
## 4 — Parquet

- Raw columnar file format — **no catalog, no ACID, no versioning**
- Fastest reads; baseline to compare against managed table formats
- Unique feature: **column pruning** and **predicate pushdown** at file level

In [62]:
PARQUET_PATH = ".tmp/local_parquet_warehouse/my_database/orders_parquet"

orders_df.write \
    .mode("overwrite") \
    .parquet(PARQUET_PATH)

print(f"Written to Parquet: {PARQUET_PATH}")

Written to Parquet: .tmp/local_parquet_warehouse/my_database/orders_parquet


In [63]:
parquet_df = spark.read.parquet(PARQUET_PATH)
parquet_df.show(truncate=False)
print(f"Rows: {parquet_df.count()}")

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10      |James Davis  |Microphone  |1       |199.99

---
## Summary

| Format | ACID | Time Travel / History | Schema Evolution | Upserts | Catalog |
|--------|------|----------------------|------------------|---------|--------|
| **Iceberg** | Yes | Snapshots | Yes | Yes (MOR) | `iceberg_catalog` |
| **Delta** | Yes | Transaction log | Yes | Yes | `spark_catalog` |
| **Hudi** | Yes | Commit timeline | Yes | Yes (COW/MOR) | Path-based |
| **Parquet** | No | No | No | No | Path-based |